## Best practices for long-running jobs on Colab

- Save intermediate checkpoints to Drive frequently.
- Use conservative thread counts via `OMP_NUM_THREADS` / `MKL_NUM_THREADS`.
- For very long runs consider using a VM/GCP instance with sustained compute quotas rather than Colab.
- Keep outputs small and upload compressed archives to Drive.
- If the runtime disconnects, restart and re-run the notebook cells that mount Drive and restore state from Drive.

In [ ]:
# Cell: Archive results
!zip -r /content/drive/MyDrive/aqrs-results/aqrs_results_$(date +%F_%T).zip /content/drive/MyDrive/aqrs-results || true
print('Zipped results in Drive folder')

## Collect outputs and save results to Drive

Compress and save results folder to Drive for later download.

In [ ]:
# Cell: Monitor resources
import psutil, time
print('CPU %:', psutil.cpu_percent(interval=1))
print('Memory:', psutil.virtual_memory())
!nvidia-smi || true

# tail the log file example
!tail -n 50 /content/drive/MyDrive/aqrs-results/feature_selector.log || true

## Monitor processes and resource usage

Use `psutil` to monitor CPU/RAM, and `nvidia-smi` for GPU. You can also stream logs back to Drive.

In [ ]:
# Cell: background execution example
# Start tests or scripts in background and write logs to Drive
!nohup python -u /content/repo/tools/colab/run_command.py --cmd "python -m pytest -q tests/test_feature_selector.py -q" > /content/drive/MyDrive/aqrs-results/feature_selector.log 2>&1 &
print('Started background job; check /content/drive/MyDrive/aqrs-results/feature_selector.log')
# show running python processes
!ps aux | grep python | grep -v grep

## Run external scripts in background and log to Drive

Example using `nohup` and background `&`. Use this to start long-running jobs and disconnect.

In [ ]:
# Cell: concurrent.futures example
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def io_task(i):
    # placeholder I/O-bound task
    import time
    time.sleep(0.5)
    return i

with ThreadPoolExecutor(max_workers=4) as ex:
    res = list(ex.map(io_task, range(8)))
print('thread results len:', len(res))

with ProcessPoolExecutor(max_workers=2) as ex:
    res2 = list(ex.map(io_task, range(4)))
print('process results len:', len(res2))

# save
import json
with open('/content/drive/MyDrive/aqrs-results/concurrent_results.json','w') as f:
    json.dump({'thread': len(res), 'proc': len(res2)}, f)
print('Saved concurrent_results.json')

## Run tasks with concurrent.futures (example)

ThreadPoolExecutor for I/O-bound tasks and ProcessPoolExecutor for CPU-bound tasks.

In [ ]:
# Cell: multiprocessing.Pool example
from multiprocessing import Pool
import time

def worker(i):
    # example CPU-bound job (replace with your function)
    s = 0
    for _ in range(10_000_00):
        s += _ % 7
    return i, s

if __name__ == '__main__':
    with Pool(processes=2) as p:
        results = p.map(worker, range(4))
    print('results sample:', results[:3])
    
# Save results
import json
with open('/content/drive/MyDrive/aqrs-results/multiproc_results.json','w') as f:
    json.dump({'results': results}, f)
print('Saved multiproc_results.json')

## Run tasks with multiprocessing.Pool (example)

Example: run a CPU-bound function across multiple processes and save results to Drive.

## Prepare Python virtual env / kernels (optional)

Colab runs in isolated environments. Usually `pip install` into the runtime is sufficient. If you prefer a venv or conda environment, include instructions here. We'll skip venv by default.

In [ ]:
# Cell: Configure parallelism
import os
# Choose a conservative thread count to keep CPU usage low (e.g., 2 or 4)
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['NUMEXPR_NUM_THREADS'] = '2'
print('Set OMP/MKL/NUMEXPR threads to 2')

# Verify
import psutil
print('logical CPUs:', psutil.cpu_count(logical=True))
print('physical cores:', psutil.cpu_count(logical=False))

## Set environment variables for parallelism

Set `OMP_NUM_THREADS`, `MKL_NUM_THREADS`, and `NUMEXPR_NUM_THREADS` to control thread usage by native libraries (numpy/scipy/sktime etc.). Adjust to a small number to limit CPU usage.

In [ ]:
# Cell: Detect hardware
!nvidia-smi || echo 'No NVIDIA GPU detected'

import tensorflow as tf
print('TF GPUs:', tf.config.list_physical_devices('GPU'))

# CPU cores
import os
print('CPU count:', os.cpu_count())

## Detect hardware (CPU / GPU / TPU)

Check GPUs and TPUs; choose appropriate runtime in Colab (Runtime -> Change runtime type).

In [ ]:
# Cell: Clone repo (if not present)
import os
repo_path = '/content/repo'
if not os.path.exists(repo_path):
    !git clone https://github.com/mb6226/autonomous-quant-research-system.git /content/repo
else:
    print('Repo already exists at', repo_path)

# set workdir
%cd /content/repo
print('Current dir:', !pwd)

## Sync project folder

Options:
- Clone from GitHub into `/content/repo`
- Use `rclone` to sync from Google Drive
- Use `rsync` when you have SSH access

Below we clone the GitHub repo into `/content/repo` if the folder doesn't already exist.

In [ ]:
# Cell: Install requirements (CPU)
!pip install -r /content/repo/requirements.txt

# Optional: GPU versions (uncomment if you know the wheels you need)
# !pip install xgboost==1.7.6
# !pip install lightgbm

# quick check
import sys
print('python:', sys.executable)
import pkgutil
for pkg in ('numpy','pandas','sklearn','xgboost','lightgbm'):
    print(pkg, 'installed=' , pkgutil.find_loader(pkg) is not None)

## Install required packages

This will install Python dependencies from `requirements.txt`. If you need GPU versions of XGBoost/LightGBM, uncomment and install the GPU wheels instead.

Notes:
- Some packages may require system libraries and may not compile easily on Colab.
- If install fails, consider installing specific wheels (e.g., `xgboost==1.7.6` for CPU) or using conda on Colab (via `condacolab`).

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
print('Mounting Google Drive...')
drive.mount('/content/drive')

# create results folder
import os
os.makedirs('/content/drive/MyDrive/aqrs-results', exist_ok=True)
print('Results will be saved to /content/drive/MyDrive/aqrs-results')

# Run project on Google Colab

This notebook sets up the repository on Google Colab, installs dependencies, synchronizes with Google Drive, detects hardware, configures parallelism, and provides examples to run and monitor CPU/GPU-bound tasks. Save outputs into `/content/drive/MyDrive/aqrs-results/`.

Instructions: open in Colab, run cells sequentially, edit the command cells to run the scripts or tests you want.

In [ ]:
# Cell: Start FeatureSelector safely via colab_launcher
# This uses the colab-only launcher which will refuse to run on your laptop.
# It launches the existing run_command.py via the launcher and writes logs to Drive.

!nohup python /content/repo/tools/colab/colab_launcher.py --cmd "python -u /content/repo/tools/colab/run_command.py --cmd 'python -m pytest -q tests/test_feature_selector.py -q'" > /content/drive/MyDrive/aqrs-results/feature_selector_safe.log 2>&1 &
print('Started safe background job; check /content/drive/MyDrive/aqrs-results/feature_selector_safe.log')
!ps aux | grep colab_launcher.py | grep -v grep || true

In [ ]:
# Cell: Safe FeatureSelector runner (sampled)
# Runs the `run_feature_selector.py` through the colab launcher. Default samples 30% to limit CPU.
!nohup python /content/repo/tools/colab/colab_launcher.py --cmd "python -u /content/repo/tools/colab/run_feature_selector.py --sample-frac 0.3 --push" > /content/drive/MyDrive/aqrs-results/feature_selector_run.log 2>&1 &
print('Launched sampled FeatureSelector job; logs: /content/drive/MyDrive/aqrs-results/feature_selector_run.log')
!ps aux | grep run_feature_selector.py | grep -v grep || true